In [52]:
from langchain_openai.chat_models import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.types import Send
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import DocumentStream

import math
import operator
from typing import TypedDict, List, Optional, Annotated
from pydantic import BaseModel
import requests
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
)
import os

from readability import Document
from markdownify import markdownify as md

from dotenv import load_dotenv
load_dotenv('../.env')
EMAIL_ADDRESS = os.environ.get('EMAIL_ADDRESS')

# PDF Conversion

In [2]:
# The PDF file containing the research paper
#from docling.document_converter import DocumentConverter

#converter = DocumentConverter()
#result = converter.convert("../data/simucell3d-nat-comp-sci-paper.pdf")

#markdown = result.document.export_to_markdown()

with open("converted_pdf.md", "r") as file:
    markdown = file.read()

print(markdown)


## Resource

https://doi.org/10.1038/s43588-024-00620-9

## SimuCell3D: three-dimensional simulation of tissue mechanics with cell polarization

Received: 4 April 2023

Accepted: 8 March 2024

Published online: 9 April 2024

Check for updates

Steve Runser 1,2 , Roman Vetter 1,2 &amp; Dagmar Iber 1,2

The three-dimensional (3D) organization of cells determines tissue function and integrity, and changes markedly in development and disease. Cell-based simulations have long been used to define the underlying mechanical principles. However, high computational costs have so far limited simulations to either simplified cell geometries or small tissue patches. Here, we present SimuCell3D, an efficient open-source program to simulate large tissues in three dimensions with subcellular resolution, growth, proliferation, extracellular matrix, fluid cavities, nuclei and non-uniform mechanical properties, as found in polarized epithelia. Spheroids, vesicles, sheets, tubes and other tissue geometrie

# References Extraction

In [3]:
# Create the model
llm_low_temp = ChatOpenAI(model="gpt-5-nano", temperature=0)

In [117]:
class Reference(BaseModel):
    ref_id: int
    journal: str
    title: str
    authors: Optional[List[str]] = None
    publication_year: Optional[int]
    doi: Optional[str] = ""

    # The URL that points to the paper landing page.
    html_url: Optional[str] = ""

    # The URL pointing directly to the pdf of the paper (better for data extraction)
    pdf_url: Optional[str] = ""

    is_open_access: Optional[bool] = None

    # The article content in markdown format
    content: Optional[str] = ""


class ReferenceExtractionState(TypedDict):

    # The document in markdown format
    document: str

    # Parsed and enriched references (replaced by downstream nodes).
    references: List[Reference]


class ReferenceState(TypedDict):
    reference : Reference

In [118]:
def reference_extraction_node(state: ReferenceExtractionState) -> ReferenceExtractionState:
    """
    Extract references from the bibliography section.
    """

    document = state.get("document", "")

    class ReferenceList(BaseModel):
        references: List[Reference]

    structured_llm = llm_low_temp.with_structured_output(ReferenceList)

    system_prompt = """
You are an expert at analyzing the references in scientific articles. Your task is to find all the references
in the bibliography section of the paper given to you and transcribe them.

Rules:
- do not hallucinate references
- no commentary
- no markdown
"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=document),
    ]

    res = structured_llm.invoke(messages)
    return {"references": res.references}


def fetch_crossref_metadata(ref: Reference) -> Reference:
    """
        Get the following metadata about the paper from crossref:
        - author names
        - journal name
        - html url
        - doi
    """
    crossref_request_params = {
                "query.title": ref.title,
                "rows": 5,
    }
    if ref.publication_year is not None:
        crossref_request_params["filter"] = (
            f"from-pub-date:{ref.publication_year}-01-01,"
            f"until-pub-date:{ref.publication_year}-12-31"
        )

    crossref_response = requests.get(
        "https://api.crossref.org/works",
        params=crossref_request_params,
        timeout=20,
    ).json()

    if crossref_response.get("message") and crossref_response["message"].get("items"):
        crossref_item = crossref_response["message"]["items"][0]

        # Get the URL to the paper
        ref.html_url = crossref_item.get("URL") or ref.html_url

        # Get its DOI to properly identify it
        ref.doi = crossref_item.get("DOI") or ref.doi

        # Get the journal name
        ref.journal = (
            (crossref_item.get("container-title") or [ref.journal])[0]
            if isinstance(crossref_item.get("container-title"), list)
            else crossref_item.get("container-title") or ref.journal
        )

        # Get the full author list, if available
        authors = []
        for author in crossref_item.get("author", []):
            given = author.get("given", "").strip()
            family = author.get("family", "").strip()
            full_name = f"{given} {family}".strip()
            if full_name:
                authors.append(full_name)
        if authors:
            ref.authors = authors
    return ref


def fetch_openaccess_metadata(ref: Reference) -> Reference:
    """
    Get the following metadata from unpaywall API:
    - is_openaccess
    - url_pdf (A URL directly pointing to the pdf download endpoint)
    """
    if not ref.doi:
        return ref

    unpaywall_url = f"https://api.unpaywall.org/v2/{ref.doi}?email={EMAIL_ADDRESS}"
    unpaywall_response = requests.get(unpaywall_url, timeout=20).json()

    ref.is_open_access = bool(unpaywall_response.get("is_oa", False))

    # Use OA location if present
    best_oa_location = unpaywall_response.get("best_oa_location") or {}
    if ref.is_open_access:
        ref.pdf_url = best_oa_location.get("url_for_pdf") or ref.pdf_url
        ref.html_url = best_oa_location.get("url_for_landing_page") or ref.html_url

    return ref


def fetch_paper_content(ref: Reference) -> Reference:
    """Load the content of the paper and save it as markdown file"""

    # Only load the content of openaccess articles
    if ref.is_open_access:

        # If we have access to the pdf download endpoint
        if ref.pdf_url:
            pdf_request_response = requests.get(ref.pdf_url, timeout=30)
            if pdf_request_response.status_code == 200:
                pdf_stream = BytesIO(pdf_request_response.content)
                converter = DocumentConverter()
                result = converter.convert(DocumentStream(name="paper.pdf", stream=pdf_stream))
                content = result.document.export_to_markdown()
                if len(content) > 5000:
                    ref.content = content

        # Instead try to download the article content directly from the HTML page
        if (not ref.content) and ref.html_url:
            html_request_response = requests.get(ref.html_url, timeout=30)

            if html_request_response.status_code == 200:
                doc = Document(html_request_response.text)
                article_html = doc.summary()
                content = md(article_html)
                if len(content) > 5000:
                    ref.content = content

    return ref


def fetch_reference_metadata_online_node(state: ReferenceExtractionState) -> ReferenceExtractionState:
    """
    Enrich parsed references and return replacement list.
    """

    def enrich_reference(ref: Reference) -> Optional[Reference]:
        try:
            ref = fetch_crossref_metadata(ref)
            ref = fetch_openaccess_metadata(ref)
            ref = fetch_paper_content(ref)
            return ref
        except Exception as e:
            print("Problem with reference:", ref.ref_id, "\n", type(e).__name__, e)
            return None

    enriched_references: List[Reference] = []
    references = state.get("references", [])

    if not references:
        return {"references": []}

    for ref in references:
        enriched_reference = enrich_reference(ref)
        enriched_references.append(enriched_reference)

    enriched_references.sort(key=lambda r: r.ref_id)
    return {"references": enriched_references}

In [119]:
builder = StateGraph(ReferenceExtractionState)
builder.add_node("reference_extraction_node", reference_extraction_node)
builder.add_node("fetch_reference_metadata_online_node", fetch_reference_metadata_online_node)
builder.add_edge(START, "reference_extraction_node")
builder.add_edge("reference_extraction_node", "fetch_reference_metadata_online_node")
builder.add_edge("fetch_reference_metadata_online_node", END)
graph = builder.compile()

In [120]:
initial_state: ReferenceExtractionState = {
    "document": markdown,
    "references": [],
}
res = graph.invoke(initial_state)
print(res)

{'document': "## Resource\n\nhttps://doi.org/10.1038/s43588-024-00620-9\n\n## SimuCell3D: three-dimensional simulation of tissue mechanics with cell polarization\n\nReceived: 4 April 2023\n\nAccepted: 8 March 2024\n\nPublished online: 9 April 2024\n\nCheck for updates\n\nSteve Runser 1,2 , Roman Vetter 1,2 &amp; Dagmar Iber 1,2\n\nThe three-dimensional (3D) organization of cells determines tissue function and integrity, and changes markedly in development and disease. Cell-based simulations have long been used to define the underlying mechanical principles. However, high computational costs have so far limited simulations to either simplified cell geometries or small tissue patches. Here, we present SimuCell3D, an efficient open-source program to simulate large tissues in three dimensions with subcellular resolution, growth, proliferation, extracellular matrix, fluid cavities, nuclei and non-uniform mechanical properties, as found in polarized epithelia. Spheroids, vesicles, sheets, tu

# Statements Extraction

In [32]:
class Statement(BaseModel):
    
    # The sentence where the claim is made
    claim : str

    # The citation associated with the statement
    citations : List[int]

    # Explain why the citations support or not the claim
    verification_result : str | None = None
    is_verified : bool = False

    # If the verification fails, indicate why here
    verification_failure_reason : str | None = None


class StatementExtractionState(TypedDict):

    # The full scientific paper in markdown format
    document : str

    # The document chunked in different pieces to then parallelize the statement
    # extraction process
    n_chunks : int
    document_chunks : List[str]

    # The statement in the proper format
    statements: Annotated[List[Statement], operator.add]


class DocumentChunkState(TypedDict):
    document_chunk : str


In [34]:
def document_chunking_node(state: StatementExtractionState) -> StatementExtractionState:
    """
    Split the document in several chunks to then parallelize and speed up the statement
    extraction process
    """

    text = state["document"]
    target_chunks = state.get("n_chunks", 10)

    # estimate chunk size
    chunk_size = math.ceil(len(text) / target_chunks)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size * 0.1),
        separators=[
            "\n\n",   # paragraph
            "\n",
            ". ",
            "? ",
            "! ",
            " "
        ]
    )
    chunks = splitter.split_text(text)
    return {"document_chunks": chunks}

def fan_out_statement_extraction_node(state: StatementExtractionState):
    """Break the initial document in chunks to parallelize the statement extraction"""
    return [Send("statement_extraction_node", {"document_chunk": chunk}) for chunk in state["document_chunks"]]

def statement_extraction_node(state: DocumentChunkState) -> StatementExtractionState:
    """Extract from each document chunk the statements made and their associated citations"""
    chunk = state["document_chunk"]

    class StatementList(BaseModel):
        statements: List[Statement]

    structured_llm = llm_low_temp.with_structured_output(StatementList)
    prompt = f"""
        Extract all scientific claims from the following document chunk.

        For each claim:
        - extract the claim sentence
        - extract citation numbers referenced in the sentence

        Document chunk:
        {chunk}
        Rules:
        - output one claim per line
        - only include claims that have at least one citation number
        - preserve the exact citation numbers from the text
        - do not invent citations
        - do not include brackets or parentheses around citation numbers
        - do not output any text before or after the list
        - no explanations
        - no markdown
        - no numbering
    """

    result = structured_llm.invoke(prompt)
    return {"statements": result.statements}

    

In [35]:
statement_extraction_graph_builder = StateGraph(StatementExtractionState)
statement_extraction_graph_builder.add_node("document_chunking_node", document_chunking_node)
statement_extraction_graph_builder.add_node("statement_extraction_node", statement_extraction_node)

statement_extraction_graph_builder.add_edge(START, "document_chunking_node")
statement_extraction_graph_builder.add_conditional_edges("document_chunking_node", fan_out_statement_extraction_node)
statement_extraction_graph_builder.add_edge("statement_extraction_node", END)
statement_extraction_graph = statement_extraction_graph_builder.compile()


In [36]:
statement_extraction_input = {
    "n_chunks" : 10,
    "document" : markdown,
}

statement_extraction_res = statement_extraction_graph.invoke(statement_extraction_input)
print(statement_extraction_res)

{'document': "## Resource\n\nhttps://doi.org/10.1038/s43588-024-00620-9\n\n## SimuCell3D: three-dimensional simulation of tissue mechanics with cell polarization\n\nReceived: 4 April 2023\n\nAccepted: 8 March 2024\n\nPublished online: 9 April 2024\n\nCheck for updates\n\nSteve Runser 1,2 , Roman Vetter 1,2 &amp; Dagmar Iber 1,2\n\nThe three-dimensional (3D) organization of cells determines tissue function and integrity, and changes markedly in development and disease. Cell-based simulations have long been used to define the underlying mechanical principles. However, high computational costs have so far limited simulations to either simplified cell geometries or small tissue patches. Here, we present SimuCell3D, an efficient open-source program to simulate large tissues in three dimensions with subcellular resolution, growth, proliferation, extracellular matrix, fluid cavities, nuclei and non-uniform mechanical properties, as found in polarized epithelia. Spheroids, vesicles, sheets, tu

# Statements Verification

In [ ]:
# 